# Notebook E — Generalization Testing

This notebook tests whether the classifiers trained in notebooks 03a/03b generalize beyond their training conditions. Two types of generalization are evaluated:

1. **Cross-cell-line**: a model trained on MCF7 cells is applied to HCC1806 cells (and vice versa). Same technology (DropSeq), different biology.
2. **Cross-technology**: a model trained on DropSeq data is applied to SmartSeq data from the same cell line. Same biology, different sequencing platform.

For each scenario we compare the cross-condition accuracy to the in-distribution (same cell line, same technology) performance and report the drop.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

sns.set_theme(style='whitegrid')
MODELS_DIR  = '../outputs/models'
DATA_DROPSEQ = '../data/DropSeq'
DATA_SMARTSEQ = '../data/SmartSeq'

## 1. Load Models and Gene Lists

The `.pkl` models expect exactly the 500 genes that were selected during training. We load those gene lists from the CSV files saved by notebooks 03a/03b.

In [ ]:
def load_model(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# Load all saved models
models = {
    'MCF7': {
        'LogisticRegression': load_model(f'{MODELS_DIR}/MCF7_DropSeq_LogisticRegression.pkl'),
        'SVM':                load_model(f'{MODELS_DIR}/MCF7_DropSeq_SVM.pkl'),
        'RandomForest':       load_model(f'{MODELS_DIR}/MCF7_DropSeq_RandomForest.pkl'),
        'GradientBoosting':   load_model(f'{MODELS_DIR}/MCF7_DropSeq_GradientBoosting.pkl'),
    },
    'HCC1806': {
        'LogisticRegression': load_model(f'{MODELS_DIR}/HCC1806_DropSeq_Logistic Regression.pkl'),
        'SVM':                load_model(f'{MODELS_DIR}/HCC1806_DropSeq_SVM.pkl'),
        'RandomForest':       load_model(f'{MODELS_DIR}/HCC1806_DropSeq_Random Forest.pkl'),
        'GradientBoosting':   load_model(f'{MODELS_DIR}/HCC1806_DropSeq_Gradient Boosting.pkl'),
    },
}

# Gene sets used during training
mcf7_genes   = pd.read_csv('../outputs/MCF7_top_genes.csv',   index_col=0).index.tolist()
hcc1806_genes = pd.read_csv('../outputs/HCC1806_top_genes.csv', index_col=0).index.tolist()

print(f'MCF7 training genes:    {len(mcf7_genes)}')
print(f'HCC1806 training genes: {len(hcc1806_genes)}')
print(f'Genes in common: {len(set(mcf7_genes) & set(hcc1806_genes))}')

## 2. Helper Functions

In [ ]:
def load_dropseq_train(cell_line):
    """Load DropSeq train matrix for a cell line. Returns X (cells x genes) and y (labels)."""
    path = f'{DATA_DROPSEQ}/{cell_line}_Filtered_Normalised_3000_Data_train.txt'
    raw = pd.read_csv(path, sep=' ', index_col=0).T  # transpose: cells x genes
    y = raw.index.str.split('_').str[-1].values
    return raw, y


def select_genes(df, gene_list):
    """
    Subset df columns to gene_list, in that order.
    Genes missing from df are filled with 0 (not expressed / dropout).
    """
    available = [g for g in gene_list if g in df.columns]
    missing   = [g for g in gene_list if g not in df.columns]
    result = df[available].copy()
    for g in missing:
        result[g] = 0.0
    return result[gene_list].values  # return in original training order


def evaluate(model, X, y, label):
    """Run model on X and print accuracy metrics."""
    y_pred = model.predict(X)
    acc  = accuracy_score(y, y_pred)
    bacc = balanced_accuracy_score(y, y_pred)
    print(f'[{label}]')
    print(f'  Accuracy:          {acc:.4f}')
    print(f'  Balanced accuracy: {bacc:.4f}')
    print(classification_report(y, y_pred, target_names=sorted(set(y))))
    return {'label': label, 'accuracy': acc, 'balanced_accuracy': bacc}

## 3. In-Distribution Baseline

First we confirm in-distribution performance by re-applying each model to the data it was trained on (using out-of-fold CV from notebook 03 is better, but this provides a quick sanity check).

In [ ]:
results = []

for cell_line, gene_list in [('MCF7', mcf7_genes), ('HCC1806', hcc1806_genes)]:
    df_train, y_train = load_dropseq_train(cell_line)
    X_train = select_genes(df_train, gene_list)
    for model_name, model in models[cell_line].items():
        r = evaluate(
            model, X_train, y_train,
            label=f'IN-DIST | train={cell_line} DropSeq | test={cell_line} DropSeq | {model_name}'
        )
        r['train_cell'] = cell_line
        r['test_cell']  = cell_line
        r['scenario']   = 'in-distribution'
        r['model']      = model_name
        results.append(r)

## 4. Cross-Cell-Line Generalization

We apply each MCF7-trained model to HCC1806 data, and each HCC1806-trained model to MCF7 data.

The models expect the same 500 gene features they were trained on. We select those genes from the target cell line's data. Genes absent from the target data (due to different biology / dropout) are filled with 0.

In [ ]:
cross_cell_pairs = [
    ('MCF7',    mcf7_genes,    'HCC1806'),
    ('HCC1806', hcc1806_genes, 'MCF7'),
]

for train_cell, gene_list, test_cell in cross_cell_pairs:
    df_test, y_test = load_dropseq_train(test_cell)  # train file has labels
    X_test = select_genes(df_test, gene_list)
    genes_present = sum(1 for g in gene_list if g in df_test.columns)
    print(f'Cross-cell | {train_cell}→{test_cell}: {genes_present}/{len(gene_list)} training genes present in target data')
    for model_name, model in models[train_cell].items():
        r = evaluate(
            model, X_test, y_test,
            label=f'CROSS-CELL | train={train_cell} DropSeq | test={test_cell} DropSeq | {model_name}'
        )
        r['train_cell'] = train_cell
        r['test_cell']  = test_cell
        r['scenario']   = 'cross-cell'
        r['model']      = model_name
        results.append(r)

## 5. Cross-Technology Generalization

We apply the DropSeq-trained models to SmartSeq data from the same cell lines. The SmartSeq data covers the full transcriptome (~22 000 genes), so the 500 DropSeq training genes should be available — but they may look very different due to protocol differences (full-length vs 3′-end sequencing, different normalization).

In [ ]:
def load_smartseq_labeled(cell_line):
    """
    Load SmartSeq data and extract Hypoxia/Normoxia labels from column names.
    The SmartSeq matrices are genes x cells, so we transpose.
    Labels are embedded in cell names as 'Hypoxia' or 'Normoxia'.
    """
    # SmartSeq uses the same whitespace-separated format
    # Filename convention may vary; adjust if needed
    import re
    fname = f'{DATA_SMARTSEQ}/{cell_line}_SmartSeq_Filtered_Normalised_Data.txt'
    raw = pd.read_csv(fname, sep=r'\s+', engine='python', index_col=0).T  # cells x genes
    # Extract label from cell name (longest match wins)
    pattern = re.compile(r'Hypoxia|Normoxia', re.IGNORECASE)
    labels = []
    for name in raw.index:
        m = pattern.search(name)
        labels.append(m.group(0).capitalize() if m else 'Unknown')
    y = np.array(labels)
    valid = y != 'Unknown'
    return raw[valid], y[valid]


cross_tech_pairs = [
    ('MCF7',    mcf7_genes),
    ('HCC1806', hcc1806_genes),
]

for cell_line, gene_list in cross_tech_pairs:
    df_ss, y_ss = load_smartseq_labeled(cell_line)
    X_ss = select_genes(df_ss, gene_list)
    genes_present = sum(1 for g in gene_list if g in df_ss.columns)
    print(f'Cross-tech | {cell_line} DropSeq→SmartSeq: {genes_present}/{len(gene_list)} training genes present')
    for model_name, model in models[cell_line].items():
        r = evaluate(
            model, X_ss, y_ss,
            label=f'CROSS-TECH | train={cell_line} DropSeq | test={cell_line} SmartSeq | {model_name}'
        )
        r['train_cell'] = cell_line
        r['test_cell']  = cell_line
        r['scenario']   = 'cross-tech'
        r['model']      = model_name
        results.append(r)

## 6. Summary and Visualization

We compare balanced accuracy across all three scenarios (in-distribution, cross-cell-line, cross-technology).

In [ ]:
results_df = pd.DataFrame(results)[['model', 'scenario', 'train_cell', 'test_cell', 'accuracy', 'balanced_accuracy']]
results_df = results_df.sort_values(['model', 'scenario'])
print(results_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
palette = {'in-distribution': '#2a9d8f', 'cross-cell': '#e76f51', 'cross-tech': '#e9c46a'}

pivot = results_df.pivot_table(
    values='balanced_accuracy',
    index='model',
    columns='scenario',
    aggfunc='mean',
)
pivot[['in-distribution', 'cross-cell', 'cross-tech']].plot(
    kind='bar', ax=ax,
    color=[palette[c] for c in ['in-distribution', 'cross-cell', 'cross-tech']],
    edgecolor='white',
    width=0.7,
)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Generalization: balanced accuracy by scenario and model')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Scenario')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../outputs/figures/generalization_summary.png', dpi=150)
plt.show()

In [ ]:
# Performance drop relative to in-distribution
in_dist = results_df[results_df['scenario'] == 'in-distribution'].groupby('model')['balanced_accuracy'].mean()
cross    = results_df[results_df['scenario'] != 'in-distribution'].copy()
cross['in_dist_bacc'] = cross['model'].map(in_dist)
cross['drop'] = cross['in_dist_bacc'] - cross['balanced_accuracy']

print('Performance drop (in-distribution balanced accuracy − cross-condition balanced accuracy):')
print(cross[['model', 'scenario', 'train_cell', 'test_cell', 'balanced_accuracy', 'in_dist_bacc', 'drop']]
      .sort_values(['scenario', 'model']).to_string(index=False))

## 7. Interpretation

**What to look for:**

- A **small drop** (<0.05 balanced accuracy) from in-distribution to cross-condition means the model learned a signal that is genuinely shared — either across cell lines (a common hypoxia response) or across platforms (a technology-robust signal). This is the best possible outcome: the classifier captured real biology, not technical artifacts.

- A **large drop** (>0.15) means the model overfit to its training conditions. For cross-cell-line failures, the model memorized cell-line-specific patterns rather than the hypoxia signature. For cross-technology failures, the model latched onto technology-specific noise (e.g. 3′ bias in DropSeq, full-length signal in SmartSeq) rather than the underlying expression differences.

- **Cross-technology generalization is typically harder** than cross-cell-line because DropSeq and SmartSeq have fundamentally different noise profiles, dynamic ranges, and gene coverage. A model that succeeds at cross-technology generalization is particularly strong evidence that it learned something biologically real.

*(Fill in observed findings here after running the notebook.)*